# OOP Week 7 -- SOLID Principles: SRP & OCP

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-6
**Focus:** Single Responsibility Principle, Open/Closed Principle

---

## Learning Objectives

1. State the Single Responsibility Principle (SRP) in your own words
2. Identify SRP violations and refactor them
3. State the Open/Closed Principle (OCP)
4. Design classes that can be extended without modification
5. Recognize these principles in the pipeline architecture

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Section 1: Single Responsibility Principle (SRP)

**"A class should have one, and only one, reason to change."**
-- Robert C. Martin

In plain English: **each class should do ONE job**.

### The Restaurant Analogy

Imagine a restaurant where one person is the chef, the waiter, the cashier, AND the dishwasher. If you need to change how dishes are washed, you have to modify the same person who cooks. That is fragile and confusing.

Better: separate roles. The Chef cooks. The Waiter serves. The Cashier handles payments. Change one without affecting others.

### SRP Violation: The God Class

In [ ]:
# BAD: One class does EVERYTHING
class GodPipeline:
    def __init__(self, path, config):
        self.path = path
        self.config = config

    def load(self):
        self.data = [{"value": 25}, {"value": -5}, {"value": 30}]
        print("Loaded")

    def clean(self):
        self.data = [r for r in self.data if r.get("value", 0) >= 0]
        print("Cleaned")

    def analyze(self):
        vals = [r["value"] for r in self.data]
        self.results = {"mean": sum(vals) / len(vals)}
        print("Analyzed")

    def plot(self):
        print("Plotted")

    def export(self):
        print("Exported")

    def send_email(self):     # NOT its job!
        print("Email sent")

    def backup_database(self):  # DEFINITELY not its job!
        print("Database backed up")


# Problems:
print("GodPipeline has 7 reasons to change!")
print("Change email format? Modify GodPipeline.")
print("Change analysis? Modify GodPipeline.")
print("Change plotting? Modify GodPipeline.")
print("Everything is tangled together.")

**Expected Output:**
```
GodPipeline has 7 reasons to change!
Change email format? Modify GodPipeline.
Change analysis? Modify GodPipeline.
Change plotting? Modify GodPipeline.
Everything is tangled together.
```

### SRP Applied: Separate Components

In [ ]:
# GOOD: Each class has ONE responsibility

class DataLoader:
    """ONLY loads data."""
    def load(self, path):
        return [{"value": 25}, {"value": -5}, {"value": 30}]

class DataCleaner:
    """ONLY cleans data."""
    def clean(self, data):
        return [r for r in data if r.get("value", 0) >= 0]

class DataAnalyzer:
    """ONLY analyzes data."""
    def analyze(self, data):
        vals = [r["value"] for r in data]
        return {"mean": sum(vals) / len(vals)} if vals else {}

class DataPlotter:
    """ONLY creates plots."""
    def plot(self, data, results):
        print("Created plots")

class DataReporter:
    """ONLY exports reports."""
    def export(self, results):
        print("Exported report")


# Compose them
loader = DataLoader()
cleaner = DataCleaner()
analyzer = DataAnalyzer()

raw = loader.load("data.csv")
clean = cleaner.clean(raw)
results = analyzer.analyze(clean)

print("Each class has exactly ONE job:")
print("  DataLoader -> loads")
print("  DataCleaner -> cleans")
print("  DataAnalyzer -> analyzes")
print("  DataPlotter -> plots")
print("  DataReporter -> exports")
print("Results:", results)

**Expected Output:**
```
Each class has exactly ONE job:
  DataLoader -> loads
  DataCleaner -> cleans
  DataAnalyzer -> analyzes
  DataPlotter -> plots
  DataReporter -> exports
Results: {'mean': 27.5}
```

---
## Section 2: Open/Closed Principle (OCP)

**"Software entities should be open for extension, but closed for modification."**

In plain English: **add new behavior by writing NEW code, not by changing EXISTING code**.

### The Plugin Analogy

Your phone is 'closed' -- you do not modify the operating system. But it is 'open' -- you install new apps (plugins) that add functionality. The phone's core code never changes.

### OCP Violation: Changing Existing Code to Add Features

In [ ]:
# BAD: Must modify this function every time we add an analysis type
def analyze_bad(values, analysis_type):
    if analysis_type == "mean":
        return sum(values) / len(values)
    elif analysis_type == "std":
        m = sum(values) / len(values)
        return (sum((x-m)**2 for x in values) / len(values)) ** 0.5
    # To add 'median', we must MODIFY this function!
    # elif analysis_type == "median":
    #     ...
    else:
        raise ValueError("Unknown: " + analysis_type)

print("mean:", analyze_bad([10, 20, 30], "mean"))
print("Problem: adding 'median' requires changing existing code!")

**Expected Output:**
```
mean: 20.0
Problem: adding 'median' requires changing existing code!
```

### OCP Applied: Extend Without Modifying

In [ ]:
class AnalyzerBase:
    """Base analyzer -- extend by creating subclasses."""
    def analyze(self, values):
        raise NotImplementedError

class MeanAnalyzer(AnalyzerBase):
    def analyze(self, values):
        return {"mean": sum(values) / len(values)} if values else {}

class StdAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        m = sum(values) / len(values)
        return {"std": (sum((x-m)**2 for x in values) / len(values)) ** 0.5}

# NEW: Add median WITHOUT changing existing code!
class MedianAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        s = sorted(values)
        n = len(s)
        if n % 2 == 0:
            return {"median": (s[n//2 - 1] + s[n//2]) / 2}
        return {"median": s[n//2]}

# NEW: Event detector WITHOUT changing existing code!
class EventAnalyzer(AnalyzerBase):
    def __init__(self, threshold):
        self.threshold = threshold
    def analyze(self, values):
        events = sum(1 for v in values if v > self.threshold)
        return {"events_above": events, "threshold": self.threshold}


# Use them all
values = [10, 25, 30, 55, 20, 45, 60]
analyzers = [MeanAnalyzer(), StdAnalyzer(), MedianAnalyzer(), EventAnalyzer(40)]

all_results = {}
for a in analyzers:
    result = a.analyze(values)
    all_results.update(result)
    print(type(a).__name__ + ":", result)

print()
print("Added 2 new analyzers WITHOUT changing any existing code!")

**Expected Output:**
```
MeanAnalyzer: {'mean': 35.0}
StdAnalyzer: {'std': 16.583}
MedianAnalyzer: {'median': 30}
EventAnalyzer: {'events_above': 3, 'threshold': 40}

Added 2 new analyzers WITHOUT changing any existing code!
```

---
### Try It!

Create a `MinMaxAnalyzer(AnalyzerBase)` that returns `{"min": ..., "max": ..., "range": ...}`. Add it to the list and run again. Notice: you did NOT modify any existing class.

In [ ]:
# YOUR CODE HERE


---
### Design Decision: SRP + OCP = Composable Architecture

These two principles work together:

- **SRP** keeps each class small and focused
- **OCP** lets you add features by creating new classes

The result: a system made of small, independent parts that can be combined in new ways. This is exactly what our v3 pipeline architecture does.

---
## Section 3: SRP Case Study -- Refactoring

Here is a real refactoring example. We start with a class that violates SRP and split it into proper components.

In [ ]:
# BEFORE: One class does loading AND cleaning AND reporting
class MessyProcessor:
    def __init__(self, path):
        self.path = path
        self.data = None
        self.clean_data = None
        self.report = None

    def process(self):
        # Loading (responsibility 1)
        self.data = [{"v": 10}, {"v": -5}, {"v": 30}, {"v": None}]
        print("Loaded")

        # Cleaning (responsibility 2)
        self.clean_data = [r for r in self.data
                           if isinstance(r.get("v"), (int, float)) and r["v"] >= 0]
        print("Cleaned")

        # Reporting (responsibility 3)
        vals = [r["v"] for r in self.clean_data]
        self.report = {"count": len(vals), "mean": sum(vals)/len(vals)}
        print("Report:", self.report)

m = MessyProcessor("data.csv")
m.process()
print("\n3 responsibilities in 1 class = SRP violation!")

**Expected Output:**
```
Loaded
Cleaned
Report: {'count': 2, 'mean': 20.0}

3 responsibilities in 1 class = SRP violation!
```

### AFTER: Refactored into 3 classes

In [ ]:
# AFTER: Each class has ONE responsibility
class Loader:
    def load(self, path):
        data = [{"v": 10}, {"v": -5}, {"v": 30}, {"v": None}]
        print("Loaded " + str(len(data)) + " rows")
        return data

class Cleaner:
    def clean(self, data):
        result = [r for r in data
                  if isinstance(r.get("v"), (int, float)) and r["v"] >= 0]
        print("Cleaned: " + str(len(data)) + " -> " + str(len(result)))
        return result

class Reporter:
    def report(self, data):
        vals = [r["v"] for r in data]
        result = {"count": len(vals), "mean": sum(vals)/len(vals)} if vals else {}
        print("Report: " + str(result))
        return result

# Compose
raw = Loader().load("data.csv")
clean = Cleaner().clean(raw)
report = Reporter().report(clean)
print("\n3 classes, 3 responsibilities = SRP satisfied!")

**Expected Output:**
```
Loaded 4 rows
Cleaned: 4 -> 2
Report: {'count': 2, 'mean': 20.0}

3 classes, 3 responsibilities = SRP satisfied!
```

---
## Section 4: OCP Case Study -- Adding Features

In [ ]:
# OCP in action: add new analysis types without changing existing code

class AnalyzerBase:
    def analyze(self, values):
        raise NotImplementedError

class SumAnalyzer(AnalyzerBase):
    def analyze(self, values):
        return {"sum": sum(values)} if values else {}

class ProductAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        result = 1
        for v in values:
            result *= v
        return {"product": result}

# Each new analyzer = new class, zero changes to existing code
values = [2, 3, 5]
for a in [SumAnalyzer(), ProductAnalyzer()]:
    print(type(a).__name__ + ":", a.analyze(values))

print("\nAdded 2 analyzers. Existing code: untouched.")

**Expected Output:**
```
SumAnalyzer: {'sum': 10}
ProductAnalyzer: {'product': 30}

Added 2 analyzers. Existing code: untouched.
```

---
### Procedural vs OOP: Adding New Analysis Types

In the procedural version, every new analysis type requires modifying the existing `analyze()` function. With OCP, you create a new class and the existing code never changes.

**Procedural approach (what you did in CP1/CP2):**

In [ ]:
# PROCEDURAL: must modify existing function
def analyze(values, analysis_type):
    if analysis_type == 'mean':
        return sum(values) / len(values)
    elif analysis_type == 'sum':
        return sum(values)
    # Adding 'product' means adding ANOTHER elif here!
    # elif analysis_type == 'product':
    #     ...
    else:
        raise ValueError('Unknown: ' + analysis_type)

print(analyze([2, 3, 5], 'mean'))

**OOP approach (what we are learning now):**

In [ ]:
# OOP (OCP): new class, no changes to existing code
class ProductAnalyzer(AnalyzerBase):
    def analyze(self, values):
        result = 1
        for v in values:
            result *= v
        return {'product': result}

# Just add to the list!
analyzers.append(ProductAnalyzer())

---
### Try It!

Identify which SOLID principle is violated in this code and fix it:

```python
class UserManager:
    def create_user(self, name, email): ...
    def delete_user(self, user_id): ...
    def send_welcome_email(self, email): ...  # ???
    def generate_report(self): ...  # ???
```

In [ ]:
# YOUR CODE HERE


---
### Try It!

Refactor this code to satisfy SRP:
```python
class FileProcessor:
    def read_file(self, path): ...
    def parse_csv(self, text): ...
    def validate_data(self, data): ...
    def save_to_database(self, data): ...
    def send_notification(self, message): ...
```
How many classes should this be? What should each one do?

In [ ]:
# YOUR CODE HERE


---
### Debugging Tip: How to spot SRP violations

Ask yourself: **'If I need to change X, do I also need to change Y?'**

If changing the email format requires touching the same class that does data analysis, that class has multiple responsibilities.

**Red flags:**
- Class has more than ~5 public methods
- Class name includes 'And' (e.g., LoaderAndCleaner)
- Class has methods that do not use the same attributes
- You cannot describe the class's job in one sentence

---
## Build from Scratch Exercise

This exercise tests whether you truly understand this week's concepts. Complete it without looking at the examples above.

In [ ]:
# BUILD FROM SCRATCH:
# Take a 'God class' (one that loads, cleans, analyzes, and exports data) and refactor it into 4 separate classes. Show before and after.

# YOUR CODE HERE


In [ ]:
# TEST your build-from-scratch code:

# YOUR TESTS HERE


---
## Connect the Dots

How does this week's concept connect to previous weeks?

In [ ]:
# Review your pipeline components -- do any violate SRP? Fix them.

# YOUR ANSWER (as comments or code):


---
## Real-World Spotting

OOP patterns are everywhere in real software. Can you spot them?

In [ ]:
# Open any popular Python package (requests, flask, django) and look at their module structure. How do they apply SRP?

# YOUR ANSWER:


---
## Diagram It

Draw an ASCII class diagram for the main classes from this week. Include:
- Class names
- Key attributes
- Key methods
- Relationships (has-a, is-a)

In [ ]:
# Draw your ASCII diagram here:
# +------------------+
# |   ClassName      |
# +------------------+
# | - attribute      |
# +------------------+
# | + method()       |
# +------------------+

# YOUR DIAGRAM:


---
## Key Vocabulary

| Term | Definition |
|------|------------|
| **SRP** | Single Responsibility Principle -- one class, one job |
| **OCP** | Open/Closed Principle -- open for extension, closed for modification |
| **SOLID** | Five design principles (SRP, OCP, LSP, ISP, DIP) |
| **God class** | A class that does too many things (SRP violation) |
| **Refactor** | Restructure code without changing behavior |

---
## Recap Exercise

Without looking at the code above, try to:

In [ ]:
# 1. Write one class from this week FROM MEMORY
#    (it does not need to be perfect)

# YOUR CODE HERE


# 2. Create an instance and call at least one method

# YOUR CODE HERE


# 3. Write one test for your class

# YOUR CODE HERE


---
## What to Review Before Next Week

Before the next session, make sure you can:

1. Explain this week's main concept in your own words
2. Write a simple example from memory
3. Identify this pattern in existing code
4. Explain WHY this pattern is useful (not just HOW)

---
## Mini-Quiz

In [ ]:
# Q1: State SRP in one sentence.
# Answer: 

# Q2: State OCP in one sentence.
# Answer: 

# Q3: How does our pipeline satisfy SRP?
# Answer: 

# Q4: How does our AnalyzerBase satisfy OCP?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)